# 05 Stage-1 recall + transfer check (Phase 1a / 1b)

**Evaluation only. Decides whether to invest in Phase 1c (retraining Stage 2 on real detector
boxes + a background class).** See `docs/small_object_research_notes.md`.

Phase 0 validated the *oracle* (perfect GT boxes): small Caries 1/2/3/5 beat V6 by +0.11..+0.22.
But that assumed **perfect recall**. This notebook replaces the oracle's GT boxes with a **real
Stage 1 = V6 detector** and answers two things:

- **Phase 1a — the gate (cheap, no retraining):** how many GT lesions does V6 actually *localize*
  (class-agnostic box recall) at a few confidence thresholds? The oracle gain is only reachable for
  the boxes V6 finds. **If V6 can't localize the small Caries, Phase 1 is capped here.**
- **Phase 1b — transfer check (diagnostic):** feed V6's predicted boxes into the **current**
  `stage2_best.pt` (trained only on tight GT boxes, **no background class**) and score the full
  V6→Stage2 pipeline with the same Mask-mAP code as `src/03`/`src/04`. Two variants:
  - *full* (all V6 boxes incl. false positives) — realistic-but-pessimistic; the model cannot reject
    FP boxes, so expect precision to suffer (this is exactly why Phase 1c adds a background class);
  - *TP-only* (keep only V6 boxes that match a GT) — simulates a perfect FP filter, isolating
    refinement quality on real boxes.

**Inputs (add as Kaggle Datasets/Models):** the V6 detector checkpoint (e.g.
`version6-...best.pt`) and the Phase-0 `stage2_best.pt`. The notebook auto-detects both by filename.

**Read it like the research note says:** decision metric is the comparable full-image AP, judged on
small Caries with adequate support (1/2/3/5); large classes route to V6; small classes are low-weight
so aggregate mAP moves modestly.


## 1. Setup

In [ ]:
# Ultralytics (Stage 1 = V6) + smp (Stage 2). Training kernels usually have Internet on.
try:
    import ultralytics, segmentation_models_pytorch  # noqa
    print("ultralytics + smp already present")
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "ultralytics", "segmentation-models-pytorch"])

In [ ]:
import os, json, math
from pathlib import Path
import numpy as np
import cv2
import yaml
import torch
import segmentation_models_pytorch as smp
from ultralytics import YOLO

cv2.setNumThreads(0)          # avoid cv2-thread contention
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

## 2. Configuration

`ROI_INPUT` / `PAD_MODE` / `PAD_FACTOR` **must match the Phase-0 run that produced `stage2_best.pt`**
(otherwise the ROI distribution shifts and the transfer check is unfair). Phase 0 used
relative padding 1.5x at 224.

In [ ]:
# --- ROI framing (MUST match the stage2_best.pt training run) ---
ROI_INPUT   = 224
PAD_MODE    = "relative"
PAD_FACTOR  = 1.5
PAD_ABS_PX  = 48
ENCODER     = "resnet18"        # architecture of stage2_best.pt

# --- Stage 1 (V6) detector inference ---
IMG_SIZE_DET   = 768            # V6 was trained at 768
CAPTURE_CONF   = 0.01           # run once at very low conf, then filter in python
DET_IOU        = 0.60           # YOLO NMS IoU
MAX_DET        = 300

# --- Phase 1a recall sweep ---
CONF_SWEEP   = [0.05, 0.10, 0.25]   # Stage-1 conf thresholds to report recall at
RECALL_IOUS  = [0.30, 0.50]         # box-IoU for "localized"; 0.3 is usable because ROI is padded

# --- Phase 1b pipeline mAP ---
PIPE_CONFS   = [0.05, 0.25]         # conf thresholds to score the full pipeline at
IOU_THRESHOLDS = np.linspace(0.5, 0.95, 10)   # mask-AP thresholds (same as src/03/04)

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

# V6 per-class Mask mAP50-95 reference (src/03) + Phase-0 oracle reference, for context.
V6_REF_AP = {"abrasion":0.6471,"crown":0.6313,"filling":0.2799,"caries 1":0.1195,
             "caries 2":0.0845,"caries 3":0.0116,"caries 4":0.0000,"caries 5":0.1097,"caries 6":0.0051}
V6_REF_MAP = 0.2099
ORACLE_REF_MAP = 0.312    # Phase-0 oracle aggregate (context only)

def _norm_class_key(nm):
    s = str(nm).lower().replace("class", "")
    return " ".join(s.split())

print("ROI", ROI_INPUT, PAD_MODE, PAD_FACTOR, "| det imgsz", IMG_SIZE_DET,
      "| conf sweep", CONF_SWEEP)

## 3. Dataset + validation ground truth

In [ ]:
INPUT_ROOT = Path("/kaggle/input")
yc = list(INPUT_ROOT.rglob("yolo_seg_train.yaml"))
if not yc:
    raise FileNotFoundError("No yolo_seg_train.yaml under /kaggle/input.")
DATA_YAML = yc[0]
dcfg = yaml.safe_load(open(DATA_YAML, encoding="utf-8"))
names = dcfg.get("names")
if isinstance(names, dict):
    names = [names[k] for k in sorted(names)]
num_classes = len(names)
dataset_root = DATA_YAML.parent
yaml_root = Path(dcfg["path"]) if dcfg.get("path") else dataset_root
if not yaml_root.is_absolute():
    yaml_root = dataset_root / yaml_root

def resolve_split(v):
    if v is None: return None
    p = Path(v)
    if p.is_absolute(): return p
    for c in (yaml_root / p, dataset_root / p):
        if c.exists(): return c
    return yaml_root / p

val_images = resolve_split(dcfg.get("val", dcfg.get("valid")))
def labels_dir_for(images_dir):
    parts = list(Path(images_dir).parts)
    if "images" in parts:
        i = len(parts) - 1 - parts[::-1].index("images"); parts[i] = "labels"
        return Path(*parts)
    return Path(images_dir).parent / "labels"

IMG_EXTS = {".jpg",".jpeg",".png",".bmp",".webp",".tif",".tiff"}
def parse_seg_line(line):
    parts = line.strip().split()
    if len(parts) < 7: return None
    try:
        cls = int(float(parts[0])); coords = [float(v) for v in parts[1:]]
    except ValueError:
        return None
    if len(coords) % 2: coords = coords[:-1]
    poly = np.asarray(coords, dtype=np.float64).reshape(-1, 2)
    return (cls, poly) if len(poly) >= 3 else None

val_objs = {}   # image_path -> list of (cls, poly_norm)
lbl_dir = labels_dir_for(val_images)
for ip in sorted(p for p in Path(val_images).iterdir() if p.suffix.lower() in IMG_EXTS):
    lp = lbl_dir / (ip.stem + ".txt")
    objs = []
    if lp.exists():
        for line in lp.read_text().splitlines():
            pr = parse_seg_line(line)
            if pr is not None: objs.append(pr)
    val_objs[str(ip)] = objs

n_gt = np.zeros(num_classes, dtype=int)
for objs in val_objs.values():
    for c, _ in objs: n_gt[c] += 1
print("classes:", names)
print("val images:", len(val_objs), "| total GT objects:", int(n_gt.sum()))
for c in range(num_classes):
    print(f"  {names[c]:>14s}: n_gt={int(n_gt[c])}")

## 4. Load Stage 1 (V6 detector) and Stage 2 (`stage2_best.pt`)

In [ ]:
# Optional hard overrides — set these to the exact /kaggle/input paths if auto-detection
# ever picks the wrong file. Leave None to auto-detect.
# Example: MANUAL_V6_PATH = "/kaggle/input/my-v6/version6_best.pt"
MANUAL_V6_PATH = None
MANUAL_S2_PATH = None

def find_pt(include, exclude=()):
    cands = []
    for p in Path("/kaggle/input").rglob("*.pt"):
        t = str(p).lower()
        if any(x in t for x in exclude): continue
        score = sum(10 for k in include if k in t) + (20 if p.name.lower().endswith("best.pt") else 0)
        if score > 0: cands.append((score, p))
    cands.sort(key=lambda z: z[0], reverse=True)
    return [p for _, p in cands]

# V6 detector = the model you trained at V6 (YOLOv8s-seg, imgsz 768). "version6_best.pt" matches
# via the "version6" keyword, and the stage2 file is excluded so the two never collide.
v6_cands = find_pt(["version6", "v6", "yolov8s", "img768"], exclude=["stage2"])
s2_cands = find_pt(["stage2"])
print("V6 candidates:");  [print("  ", p) for p in v6_cands[:5]]
print("Stage2 candidates:"); [print("  ", p) for p in s2_cands[:5]]

V6_PATH = Path(MANUAL_V6_PATH) if MANUAL_V6_PATH else (v6_cands[0] if v6_cands else None)
S2_PATH = Path(MANUAL_S2_PATH) if MANUAL_S2_PATH else (s2_cands[0] if s2_cands else None)
if V6_PATH is None or not V6_PATH.exists():
    raise FileNotFoundError("No V6 detector .pt found. Add it as a Kaggle input or set MANUAL_V6_PATH.")
if S2_PATH is None or not S2_PATH.exists():
    raise FileNotFoundError("No stage2_best.pt found. Add it as a Kaggle input or set MANUAL_S2_PATH.")
print("\nUsing V6 :", V6_PATH)
print("Using S2 :", S2_PATH)

detector = YOLO(str(V6_PATH))

stage2 = smp.Unet(encoder_name=ENCODER, encoder_weights=None, in_channels=3,
                  classes=1, activation=None,
                  aux_params=dict(classes=num_classes, dropout=0.2)).to(DEVICE)
state = torch.load(str(S2_PATH), map_location=DEVICE)
state = state.get("state_dict", state) if isinstance(state, dict) else state
missing, unexpected = stage2.load_state_dict(state, strict=False)
stage2.eval()
print("\nStage2 loaded. missing:", len(missing), "unexpected:", len(unexpected))
if missing or unexpected:
    print("  (a non-trivial mismatch means the architecture / num_classes differ from training)")

## 5. Run V6 on the validation images (capture low-conf boxes once)

One detector pass per image at `CAPTURE_CONF`; all boxes are stored so the conf sweep is just a
filter in Python (no re-inference). Stage-1 boxes are used **class-agnostically** — Stage 2
re-classifies, so only localization matters here.

In [ ]:
det_by_image = {}   # path -> list of dict(box=(x0,y0,x1,y1), conf, cls)
wh_by_image = {}
for ip in val_objs:
    img = cv2.imread(ip, cv2.IMREAD_COLOR)
    h, w = img.shape[:2]; wh_by_image[ip] = (w, h)
    res = detector.predict(source=img, imgsz=IMG_SIZE_DET, conf=CAPTURE_CONF, iou=DET_IOU,
                           max_det=MAX_DET, device=DEVICE, task="segment",
                           verbose=False, save=False)[0]
    dets = []
    if res.boxes is not None and len(res.boxes) > 0:
        xyxy = res.boxes.xyxy.cpu().numpy()
        conf = res.boxes.conf.cpu().numpy()
        cls  = res.boxes.cls.cpu().numpy().astype(int)
        for b, cf, cl in zip(xyxy, conf, cls):
            x0, y0, x1, y1 = [int(round(v)) for v in b]
            x0 = max(0, min(x0, w-1)); y0 = max(0, min(y0, h-1))
            x1 = max(x0+1, min(x1, w)); y1 = max(y0+1, min(y1, h))
            dets.append({"box": (x0, y0, x1, y1), "conf": float(cf), "cls": int(cl)})
    det_by_image[ip] = dets
tot = sum(len(d) for d in det_by_image.values())
print(f"captured {tot} V6 boxes over {len(det_by_image)} images at conf>={CAPTURE_CONF}")

## 6. Phase 1a — Stage-1 localization recall (the gate)

For each GT object, is there a V6 box (conf >= t) overlapping it by box-IoU >= thr? Class-agnostic.
This is the ceiling Stage 2 can possibly refine. Watch the **small Caries**.

In [ ]:
def gt_pixel_bbox(poly_norm, w, h):
    xs = poly_norm[:, 0] * w; ys = poly_norm[:, 1] * h
    return (max(0, int(np.floor(xs.min()))), max(0, int(np.floor(ys.min()))),
            min(w, int(np.ceil(xs.max()))), min(h, int(np.ceil(ys.max()))))

def box_iou(a, b):
    ix0, iy0 = max(a[0], b[0]), max(a[1], b[1])
    ix1, iy1 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0, ix1 - ix0), max(0, iy1 - iy0)
    inter = iw * ih
    if inter <= 0: return 0.0
    aa = (a[2]-a[0])*(a[3]-a[1]); bb = (b[2]-b[0])*(b[3]-b[1])
    return inter / (aa + bb - inter)

import pandas as pd
recall_rows = []
for iou_thr in RECALL_IOUS:
    for conf_t in CONF_SWEEP:
        hit = np.zeros(num_classes, dtype=int)
        for ip, objs in val_objs.items():
            w, h = wh_by_image[ip]
            boxes = [d["box"] for d in det_by_image[ip] if d["conf"] >= conf_t]
            for cls, poly in objs:
                gb = gt_pixel_bbox(poly, w, h)
                if any(box_iou(gb, bx) >= iou_thr for bx in boxes):
                    hit[cls] += 1
        for c in range(num_classes):
            recall_rows.append({"iou": iou_thr, "conf": conf_t, "class": names[c],
                                "n_gt": int(n_gt[c]),
                                "recall": (hit[c] / n_gt[c]) if n_gt[c] else float("nan")})
recall_df = pd.DataFrame(recall_rows)

# Pretty per-class recall table at box-IoU 0.5 (the standard gate).
print("=== Phase 1a: Stage-1 (V6) localization recall @ box-IoU 0.5 ===")
piv = recall_df[recall_df.iou == 0.5].pivot(index="class", columns="conf", values="recall")
piv = piv.reindex(names)
print(piv.round(3).to_string())
print("\n(also computed at box-IoU 0.30 — looser, still usable because ROIs are padded)")
piv30 = recall_df[recall_df.iou == 0.30].pivot(index="class", columns="conf", values="recall").reindex(names)
print(piv30.round(3).to_string())

## 7. Phase 1b — transfer check: full V6 -> Stage2 pipeline Mask mAP

Run the current `stage2_best.pt` on every captured V6 box (once, batched per image), then score the
pipeline at each conf threshold. `full` = all boxes (FPs included; the model has no background class
so it mislabels empty boxes). `TP-only` = keep only boxes matching a GT (perfect FP rejection),
isolating refinement quality. Detection score = V6 conf x Stage-2 class prob.

In [ ]:
def padded_square_box(x0, y0, x1, y1, w, h):
    bw, bh = x1 - x0, y1 - y0
    cx, cy = (x0 + x1) / 2.0, (y0 + y1) / 2.0
    half = (max(bw, bh) * (1.0 + PAD_FACTOR) / 2.0) if PAD_MODE == "relative" else (max(bw, bh) / 2.0 + PAD_ABS_PX)
    half = max(half, 1.0)
    sx0, sy0, sx1, sy1 = cx - half, cy - half, cx + half, cy + half
    sx0 = max(0.0, min(sx0, w - 1)); sy0 = max(0.0, min(sy0, h - 1))
    sx1 = max(sx0 + 1, min(sx1, w)); sy1 = max(sy0 + 1, min(sy1, h))
    return int(round(sx0)), int(round(sy0)), int(round(sx1)), int(round(sy1))

def gt_local_mask(poly_norm, w, h):
    pts = poly_norm.copy(); pts[:, 0] *= w; pts[:, 1] *= h
    gx0 = max(0, int(np.floor(pts[:,0].min()))); gy0 = max(0, int(np.floor(pts[:,1].min())))
    gx1 = min(w, int(np.ceil(pts[:,0].max())));  gy1 = min(h, int(np.ceil(pts[:,1].max())))
    gw = max(1, gx1 - gx0); gh = max(1, gy1 - gy0)
    m = np.zeros((gh, gw), dtype=np.uint8)
    pts[:, 0] -= gx0; pts[:, 1] -= gy0
    cv2.fillPoly(m, [pts.astype(np.int32)], 1)
    return (gx0, gy0, gx1, gy1), m

def iou_local(pbox, pmask, gbox, gmask):
    pa = int(pmask.sum()); ga = int(gmask.sum())
    if pa == 0 or ga == 0: return 0.0
    ix0 = max(pbox[0], gbox[0]); iy0 = max(pbox[1], gbox[1])
    ix1 = min(pbox[2], gbox[2]); iy1 = min(pbox[3], gbox[3])
    inter = 0
    if ix1 > ix0 and iy1 > iy0:
        pc = pmask[iy0-pbox[1]:iy1-pbox[1], ix0-pbox[0]:ix1-pbox[0]]
        gc = gmask[iy0-gbox[1]:iy1-gbox[1], ix0-gbox[0]:ix1-gbox[0]]
        inter = int(np.logical_and(pc, gc).sum())
    union = pa + ga - inter
    return inter / union if union > 0 else 0.0

@torch.no_grad()
def stage2_on_boxes(img, boxes):
    # Batched Stage-2 over one image's V6 boxes. Returns list of (pbox, s2_cls, s2_prob, pmask_local).
    if not boxes: return []
    h, w = img.shape[:2]; xs = []; pboxes = []
    for (x0, y0, x1, y1) in boxes:
        sb = padded_square_box(x0, y0, x1, y1, w, h); pboxes.append(sb)
        crop = cv2.resize(img[sb[1]:sb[3], sb[0]:sb[2]], (ROI_INPUT, ROI_INPUT))
        x = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        xs.append(((x - IMAGENET_MEAN) / IMAGENET_STD).transpose(2, 0, 1))
    batch = torch.from_numpy(np.stack(xs)).to(DEVICE)
    mlog, clog = stage2(batch)
    prob = torch.softmax(clog, 1).cpu().numpy()
    masks = (torch.sigmoid(mlog)[:, 0] > 0.5).cpu().numpy().astype(np.uint8)
    out = []
    for i, sb in enumerate(pboxes):
        pm = cv2.resize(masks[i], (sb[2]-sb[0], sb[3]-sb[1]), interpolation=cv2.INTER_NEAREST)
        out.append((sb, int(prob[i].argmax()), float(prob[i].max()), pm))
    return out

# Cache Stage-2 outputs for ALL captured boxes once (also record GT-match for the TP-only variant).
cache = {}   # path -> list of dict(pbox, s2_cls, s2_prob, pmask, v6_conf, matches_gt)
for ip, objs in val_objs.items():
    img = cv2.imread(ip, cv2.IMREAD_COLOR); w, h = wh_by_image[ip]
    dets = det_by_image[ip]
    s2 = stage2_on_boxes(img, [d["box"] for d in dets])
    gt_boxes = [gt_pixel_bbox(p, w, h) for _, p in objs]
    items = []
    for d, (pbox, cls, prob, pm) in zip(dets, s2):
        matches = any(box_iou(d["box"], gb) >= 0.5 for gb in gt_boxes)
        items.append({"pbox": pbox, "s2_cls": cls, "s2_prob": prob, "pmask": pm,
                      "v6_conf": d["conf"], "matches_gt": matches})
    cache[ip] = items
print("Stage-2 cached for", sum(len(v) for v in cache.values()), "boxes")

def ap_101(confs, tps, npos):
    if npos == 0: return float("nan")
    if not confs: return 0.0
    o = np.argsort(-np.asarray(confs)); tps = np.asarray(tps)[o]
    tp_c = np.cumsum(tps); fp_c = np.cumsum(1 - tps)
    rec = tp_c / npos; prec = tp_c / np.maximum(tp_c + fp_c, 1e-9)
    return sum((prec[rec >= r].max() if np.any(rec >= r) else 0.0) for r in np.linspace(0,1,101)) / 101.0

def pipeline_map(conf_t, tp_only=False):
    records = {(c, ti): [] for c in range(num_classes) for ti in range(len(IOU_THRESHOLDS))}
    for ip, objs in val_objs.items():
        w, h = wh_by_image[ip]
        gts = [(c, *gt_local_mask(p, w, h)) for c, p in objs]
        preds = []
        for it in cache[ip]:
            if it["v6_conf"] < conf_t: continue
            if tp_only and not it["matches_gt"]: continue
            preds.append((it["s2_cls"], it["v6_conf"] * it["s2_prob"], it["pbox"], it["pmask"]))
        for ti, thr in enumerate(IOU_THRESHOLDS):
            order = sorted(range(len(preds)), key=lambda i: preds[i][1], reverse=True)
            used = [False] * len(gts)
            for pi in order:
                pc, sc, pbox, pm = preds[pi]
                best, bg = 0.0, -1
                for gi, (gc, gbox, gm) in enumerate(gts):
                    if used[gi] or gc != pc: continue
                    v = iou_local(pbox, pm, gbox, gm)
                    if v > best: best, bg = v, gi
                tp = 1 if (bg >= 0 and best >= thr) else 0
                if tp: used[bg] = True
                records[(pc, ti)].append((sc, tp))
    pac = np.full(num_classes, np.nan)
    for c in range(num_classes):
        if n_gt[c] == 0: continue
        pac[c] = np.nanmean([ap_101([r[0] for r in records[(c,ti)]],
                                     [r[1] for r in records[(c,ti)]], n_gt[c])
                             for ti in range(len(IOU_THRESHOLDS))])
    return pac

pipe_results = {}
for conf_t in PIPE_CONFS:
    pipe_results[f"full@{conf_t}"] = pipeline_map(conf_t, tp_only=False)
pipe_results["TPonly@0.05"] = pipeline_map(0.05, tp_only=True)

# Report
def show(tag, pac):
    valid = ~np.isnan(pac)
    print(f"\n=== {tag} === mAP50-95 = {np.nanmean(pac[valid]):.4f}  (V6 {V6_REF_MAP:.4f}, oracle {ORACLE_REF_MAP:.3f})")
    for c in range(num_classes):
        v6 = V6_REF_AP.get(_norm_class_key(names[c]), np.nan)
        d = pac[c] - v6 if not (np.isnan(pac[c]) or np.isnan(v6)) else float("nan")
        print(f"  {names[c]:>14s} n={int(n_gt[c]):>3d}  pipe={pac[c]:.3f}  V6={v6:.3f}  d={d:+.3f}")
for tag, pac in pipe_results.items():
    show(tag, pac)

## 8. Save outputs (Kaggle Save Version)

In [ ]:
recall_csv = "/kaggle/working/phase1a_recall.csv"
recall_df.to_csv(recall_csv, index=False)

prows = []
for tag, pac in pipe_results.items():
    for c in range(num_classes):
        v6 = V6_REF_AP.get(_norm_class_key(names[c]), np.nan)
        prows.append({"variant": tag, "class": names[c], "n_gt": int(n_gt[c]),
                      "pipeline_AP": (None if np.isnan(pac[c]) else float(pac[c])),
                      "v6_ref_AP": (None if np.isnan(v6) else float(v6))})
pipe_df = pd.DataFrame(prows)
pipe_csv = "/kaggle/working/phase1b_pipeline.csv"
pipe_df.to_csv(pipe_csv, index=False)

summary = {
    "config": {"ROI_INPUT": ROI_INPUT, "PAD_MODE": PAD_MODE, "PAD_FACTOR": PAD_FACTOR,
               "IMG_SIZE_DET": IMG_SIZE_DET, "CONF_SWEEP": CONF_SWEEP, "PIPE_CONFS": PIPE_CONFS},
    "stage1_recall_iou0.5": {
        names[c]: {str(t): (None if np.isnan(r) else round(float(r), 4))
                   for t in CONF_SWEEP
                   for r in [recall_df[(recall_df.iou==0.5)&(recall_df.conf==t)&(recall_df["class"]==names[c])]["recall"].values[0]]}
        for c in range(num_classes)},
    "pipeline_mAP": {tag: round(float(np.nanmean(pac[~np.isnan(pac)])), 4) for tag, pac in pipe_results.items()},
    "v6_ref_mAP": V6_REF_MAP, "oracle_ref_mAP": ORACLE_REF_MAP,
}
with open("/kaggle/working/phase1_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Saved to /kaggle/working:")
for p in ["phase1a_recall.csv", "phase1b_pipeline.csv", "phase1_summary.json"]:
    fp = Path("/kaggle/working") / p
    print(f"  [{'OK' if fp.exists() else 'MISSING'}] {p}")
print("\n", json.dumps(summary["pipeline_mAP"], indent=2))

## 9. How to read this -> decide on Phase 1c

**The gate (Phase 1a):** look at small-Caries **recall** (1/2/3/5) at box-IoU 0.5 (and 0.3).
- If recall is decent (say >= 0.5-0.6), a real Stage 1 *can* feed Stage 2 enough small lesions →
  Phase 1c is worth it; the oracle gain is partially reachable.
- If recall is low, the pipeline is capped at Stage 1 — improving Stage 2 won't help, and the effort
  should shift to Stage-1 recall (lower conf, SAHI-assist, a dedicated small-object detector) first.

**The transfer check (Phase 1b):** `full` will likely be **worse than V6** — the current Stage 2 has
**no background class** and cannot reject V6's false-positive boxes, so precision tanks. That is
expected and is *not* a reason to abandon the approach. The informative number is **`TP-only`**:
if, on V6's true-positive boxes, Stage-2's small-Caries AP stays well above V6, then Phase 1c
(retrain on real boxes **+ a background class** to reject FPs) is justified — it should recover much
of the gap between `full` and `TP-only`.

Decision rule of thumb: **proceed to Phase 1c if (small-Caries recall is non-trivial) AND
(TP-only small-Caries AP clearly beats V6).** Otherwise fix Stage-1 recall first.
